In [ ]:
import os
import cv2 as cv2
import numpy as np
from tqdm import tqdm

In [ ]:
def stitch_sliding_patches(patch_folder, base_name, patch_size=512, save_path=None):
    import re
    
    pattern = re.compile(rf"^{re.escape(base_name)}_r(\d+)_c(\d+)_\d+\.png$")
    
    patch_infos = []
    for fn in os.listdir(patch_folder):
        m = pattern.match(fn)
        if m:
            r_start = int(m.group(1))
            c_start = int(m.group(2))
            patch_infos.append((fn, r_start, c_start))
    
    if not patch_infos:
        raise ValueError(f"No patches found for base_name={base_name}")
    
    max_r_end = max(r + patch_size for _, r, _ in patch_infos)
    max_c_end = max(c + patch_size for _, _, c in patch_infos)
    
    H, W = max_r_end, max_c_end
    
    acc = np.zeros((H, W, 3), dtype=np.float32)
    cnt = np.zeros((H, W, 1), dtype=np.float32)
    
    for fn, r_start, c_start in patch_infos:
        patch_path = os.path.join(patch_folder, fn)
        patch_bgr = cv2.imread(patch_path, cv2.IMREAD_COLOR)
        if patch_bgr is None:
            print(f"Warning: cannot read patch {patch_path}")
            continue
        
        patch = patch_bgr.astype(np.float32)
        
        r_end = r_start + patch_size
        c_end = c_start + patch_size
        
        acc[r_start:r_end, c_start:c_end, :] += patch
        cnt[r_start:r_end, c_start:c_end, :] += 1.0
    
    cnt_safe = np.where(cnt == 0, 1.0, cnt)
    acc = acc / cnt_safe
    
    stitched_img = acc.clip(0, 255).astype(np.uint8)
    
    if save_path is not None:
        cv2.imwrite(save_path, stitched_img)
    
    return stitched_img

Please enter the path of the analysis folder and output folder

In [ ]:
test_image_folder = '1;1;1;3;159;01' # original image path
output_cropped_folder = '1;1;1;3;159;01_crop' # output path for image cropping

The main code for cropping images using a sliding window

In [ ]:
patch_size = 512 # window size
stride = 256 # stride step

for image_name in tqdm(os.listdir(test_image_folder)):

    cropped_folder_name = image_name.split('.')[0]
    if not image_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
        continue

    img_path = os.path.join(test_image_folder, image_name)

    img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Warning: cannot read image {img_path}")
        continue
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    img_float = img / 255.0
    img_uint8 = (img_float * 255).clip(0, 255).astype(np.uint8)

    # ====== start sliding window cropping ====== #

    H, W = img_uint8.shape[:2]
    pad_h = max(0, patch_size - H)
    pad_w = max(0, patch_size - W)

    if pad_h > 0 or pad_w > 0:
        img_padded = np.zeros((H + pad_h, W + pad_w, 3), dtype=np.uint8)
        img_padded[:H, :W] = img_uint8
    else:
        img_padded = img_uint8

    Hp, Wp = img_padded.shape[:2]

    r_starts = list(range(0, max(Hp - patch_size + 1, 1), stride))
    c_starts = list(range(0, max(Wp - patch_size + 1, 1), stride))

    if r_starts[-1] != Hp - patch_size:
        r_starts.append(Hp - patch_size)
    if c_starts[-1] != Wp - patch_size:
        c_starts.append(Wp - patch_size)
    
    patch_idx = 0
    for r_start in r_starts:
        for c_start in c_starts:
            r_end = r_start + patch_size
            c_end = c_start + patch_size

            crop = img_padded[r_start:r_end, c_start:c_end]

            if np.any(crop):
                base_name = f"{cropped_folder_name}_r{r_start}_c{c_start}_{patch_idx}.png"
                output_path = os.path.join(output_cropped_folder, base_name)

                crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
                cv2.imwrite(output_path, crop_bgr)

                patch_idx += 1

The main code for image stitching

In [ ]:
test_image_folder = '1;1;1;3;159;01'   # original image path
predicted_mask_folder = '1;1;1;3;159;01_predicted' # the root mask path predicted by Root-TransUNet
stitch_image_folder = '1;1;1;3;159;01_stitch'  # output path for image stitching

petri_dish_image_path = os.listdir(test_image_folder)[0] # the path of a reference petri diish image
petri_dish_image_bgr = cv2.imread(petri_dish_image_path, cv2.IMREAD_COLOR)
img = cv2.cvtColor(petri_dish_image_bgr, cv2.COLOR_BGR2RGB)
        
img_float = img / 255.0
img_uint8 = (img_float * 255).clip(0, 255).astype(np.uint8)

orig_H, orig_W = img_uint8.shape[:2] # orig_H and orig_W are the dimensions of the original petri dish part

 # ====== start image stitching ====== #

base_name_list = []
for image_name in tqdm(os.listdir(predicted_mask_folder)):

    idx = image_name.find("_r")
    if idx == -1:
        continue
    prefix = image_name[:idx]
    
    if prefix not in base_name_list:
        base_name_list.append(prefix)

for base_name in base_name_list:
    recon_save_path = os.path.join(stitch_image_folder, base_name + '.png')
    
    try:
        stitched = stitch_sliding_patches(
            patch_folder=predicted_mask_folder,
            base_name=base_name,
            patch_size=512,
            save_path=None
        )
        stitched_cropped = stitched[0:orig_H, 0:orig_W]
        cv2.imwrite(recon_save_path, stitched_cropped)
    
    except Exception as e:
        print(f"Error processing {base_name} in {predicted_mask_folder}: {e}")


